In [18]:
import os
import mne
import numpy as np
import pandas as pd
import pickle
import random
from sklearn.model_selection import train_test_split

In [30]:
# Paths for input csv folders
train_data_path = 'train'
chunk_size = 3.0

labels_df = pd.read_csv(f'TrainLabels.csv')
labels_map = labels_df.set_index('IdFeedBack')['Prediction'].to_dict()

rs = 42 # seed number for data split
nchan = 56 # number of channels

channels_loc = pd.read_csv(f'ChannelsLocation.csv')

output_folders = [f's{rs}_n{nchan}/train', f's{rs}_n{nchan}/val', f's{rs}_n{nchan}/test']

wired_file = []

# Create output folders if they don't exist
for folder in output_folders:
    os.makedirs(folder, exist_ok=True)

# Function to process csv files: downsample, chunk, and label
def process_csv(df, sfreq=200,selected_channels=None):


    ch_names = [col for col in df.columns if col not in ['Time', 'FeedBackEvent', 'EOG']]
    eeg_data = df[ch_names].T.values
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_data, info)

    feedback_events = df.index[df['FeedBackEvent'] == 1].tolist()
    

    # Select only specified channels
    if selected_channels:
        raw.pick_channels(selected_channels)

    # raw.filter(l_freq=0.1, h_freq=75.0, n_jobs=-1)
    # raw.notch_filter(50.0, n_jobs=-1)

    # Downsample if necessary
    if raw.info['sfreq'] != sfreq:
        raw.resample(sfreq, n_jobs=-1)
        feedback_events = [int(idx * sfreq / raw.info['sfreq']) for idx in feedback_events]

    # signals = raw.get_data(units='uV')  # (n_channels, n_samples)
    signals = raw.get_data()

    return signals, feedback_events, ch_names

def extract_epochs(file_path, labels_map, sfreq=200, epoch_duration=chunk_size):
    
    try:
        df = pd.read_csv(file_path)
        signals, feedback_events, ch_names = process_csv(df, sfreq)
        epoch_samples = int(epoch_duration * sfreq) #800

        filename = os.path.basename(file_path)
        # filename = 'Data_S07_Sess04.csv'
        prefix = filename.replace('Data_', '').replace('.csv', '')
        
        epochs = []
        for fb_idx, event_idx in enumerate(feedback_events):

            start_idx = event_idx
            end_idx = event_idx + epoch_samples
            
            if end_idx <= signals.shape[1]:
                epoch_data = signals[:, start_idx:end_idx]
                
                epoch_id = f"{prefix}_FB{fb_idx + 1:03d}"
                
                label = labels_map.get(epoch_id, -1)
                
                if label != -1:
                    epochs.append((epoch_data, label, epoch_id))
            elif end_idx > signals.shape[1]:
                
                print(end_idx, "<-----------")
        
        print(f"extract {len(epochs)} epochs")
        if len(epochs) < 60:
            wired_file.append(filename)
        return epochs
        
    except Exception as e:
        print(f"error: {e}")
        import traceback
        traceback.print_exc()
        return []

def process_inria_bci_challenge():
    
    all_train_epochs = []
    train_files = sorted([f for f in os.listdir(train_data_path) if f.endswith('.csv')])
    
    for filename in train_files:
        file_path = os.path.join(train_data_path, filename)
        # file_path = 'train/Data_S07_Sess04.csv'
        print(filename)
        epochs = extract_epochs(file_path, labels_map)
    
        all_train_epochs.extend(epochs)
    
    train_labels = [label for _, label, _ in all_train_epochs]
    print(f"Label distribution: {pd.Series(train_labels).value_counts().to_dict()}")
    

    train_epochs, temp_epochs = train_test_split(
        all_train_epochs,
        test_size=0.2,
        random_state=rs,
        stratify=train_labels,
        shuffle=True
    )

    temp_labels = [label for _, label, _ in temp_epochs]
    val_epochs, test_epochs = train_test_split(
        temp_epochs,
        test_size=0.5,
        random_state=rs,
        stratify=temp_labels,
        shuffle=True
    )

    total = len(all_train_epochs)
    print(f"Train: {len(train_epochs)}/{total} ({len(train_epochs)/total:.1%})")
    print(f"Val: {len(val_epochs)}/{total} ({len(val_epochs)/total:.1%})")
    print(f"Test: {len(test_epochs)}/{total} ({len(test_epochs)/total:.1%})")
    
    random.seed(rs)
    random.shuffle(train_epochs)
    random.shuffle(val_epochs)
    random.shuffle(test_epochs)
    
    
    save_epochs(train_epochs, f's{rs}_n{nchan}/train')
    save_epochs(val_epochs, f's{rs}_n{nchan}/val')
    save_epochs(test_epochs, f's{rs}_n{nchan}/test')

# Function to save chunks as pickle files
def save_epochs(epochs, folder):
    for i, (epoch_data, label, epoch_id) in enumerate(epochs):
        
        sample = {
            'signal': epoch_data,  # shape: (n_channels, 800)
            'label': label,
            'epoch_id': epoch_id
        }
        
        filename = os.path.join(folder, f"{epoch_id}.pickle")
        
        with open(filename, 'wb') as f:
            pickle.dump(sample, f)
        print(f"Saved: {filename}")

if __name__ == "__main__":

    process_inria_bci_challenge()

    print("All files processed, split, and saved successfully!")
    print(wired_file)

Data_S02_Sess01.csv
Creating RawArray with float64 data, n_channels=56, n_times=132001
    Range : 0 ... 132000 =      0.000 ...   660.000 secs
Ready.
extract 60 epochs
Data_S02_Sess02.csv
Creating RawArray with float64 data, n_channels=56, n_times=128001
    Range : 0 ... 128000 =      0.000 ...   640.000 secs
Ready.
extract 60 epochs
Data_S02_Sess03.csv
Creating RawArray with float64 data, n_channels=56, n_times=127001
    Range : 0 ... 127000 =      0.000 ...   635.000 secs
Ready.
extract 60 epochs
Data_S02_Sess04.csv
Creating RawArray with float64 data, n_channels=56, n_times=128001
    Range : 0 ... 128000 =      0.000 ...   640.000 secs
Ready.
extract 60 epochs
Data_S02_Sess05.csv
Creating RawArray with float64 data, n_channels=56, n_times=196001
    Range : 0 ... 196000 =      0.000 ...   980.000 secs
Ready.
extract 100 epochs
Data_S06_Sess01.csv
Creating RawArray with float64 data, n_channels=56, n_times=132001
    Range : 0 ... 132000 =      0.000 ...   660.000 secs
Ready.
ext

In [34]:
rs = 42 # seed number for data split
nchan = 56 # number of channels
folder = f's{rs}_n{nchan}/test'
files = [f for f in os.listdir(folder) if f.endswith('.pickle')]
print(len(files))
any_file = os.path.join(folder, files[0])

with open(any_file, 'rb') as f:
        data = pickle.load(f)

signal = data['signal']
label = data['label']
epoch_id = data.get('epoch_id', 'N/A')

print(f"total file num: {len(files)}")
print(f"Epoch ID: {epoch_id}")
print(f"Signal shape: {signal.shape}") #should output: (n_channels, 800)

# print(f"range: [{signal.min():.2f}, {signal.max():.2f}] μV")
print(f"label: {label}")
print(f"signal type: {signal.dtype}")

if 'train' in folder or 'val' in folder:
    labels = []
    for f in files:
        with open(os.path.join(folder, f), 'rb') as file:
            data = pickle.load(file)
            labels.append(data['label'])
    print(f"label distribution: {pd.Series(labels).value_counts().to_dict()}")

544
total file num: 544
Epoch ID: S14_Sess03_FB006
Signal shape: (56, 600)
label: 0
signal type: float64


In [17]:
training_directory = f's{rs}_n{nchan}/train'  # Update this path to your training directory

def inspect_label_ratios(directory):
    total_labels = 0
    class_labels = {0: 0, 1: 0}  # Assuming binary labels: 0 for normal and 1 for increase

    for filename in os.listdir(directory):
        if filename.endswith('.pkl'):
            file_path = os.path.join(directory, filename)
            print(f"Processing {filename} from {directory}...")

            try:
                # Load the pickle file
                with open(file_path, 'rb') as f:
                    data_dict = pickle.load(f)

                # Check if the expected keys are in the dictionary
                if 'label' not in data_dict:
                    print(f"Warning: 'label' key not found in {filename}. Skipping this file.")
                    continue
                
                labels = data_dict['label']

                # Check the type of labels
                if isinstance(labels, list):
                    labels = np.array(labels)  # Convert list to numpy array
                elif isinstance(labels, int):
                    labels = np.array([labels])  # Convert single integer to a numpy array
                elif not isinstance(labels, np.ndarray):
                    print(f"Warning: Unexpected label format in {filename}. Skipping this file.")
                    continue

                # Count the labels
                total_labels += len(labels)
                class_labels[1] += np.sum(labels)  # Count positive class (increase)
                class_labels[0] += len(labels) - np.sum(labels)  # Count negative class (normal)

            except Exception as e:
                print(f"Error processing {filename}: {e}")

    # Calculate the ratios
    if total_labels > 0:
        ERP_ratio = class_labels[1] / total_labels
        nonERP_ratio = class_labels[0] / total_labels
    else:
        ERP_ratio = 0
        nonERP_ratio = 0

    print(f"\nLabel Ratios:")
    print(f"1 Class Ratio: {ERP_ratio:.4f} (total: {total_labels}, class count: {class_labels[1]})")
    print(f"0 Class Ratio: {nonERP_ratio:.4f} (total: {total_labels}, class count: {class_labels[0]})")

# Inspect the label ratios for all classes in the training directory
inspect_label_ratios(training_directory)

Processing S11_Sess01_FB010.pkl from s42_n56/train...
Processing S18_Sess03_FB004.pkl from s42_n56/train...
Processing S26_Sess03_FB004.pkl from s42_n56/train...
Processing S16_Sess04_FB039.pkl from s42_n56/train...
Processing S12_Sess05_FB076.pkl from s42_n56/train...
Processing S07_Sess03_FB039.pkl from s42_n56/train...
Processing S18_Sess03_FB010.pkl from s42_n56/train...
Processing S23_Sess05_FB059.pkl from s42_n56/train...
Processing S02_Sess05_FB070.pkl from s42_n56/train...
Processing S11_Sess01_FB004.pkl from s42_n56/train...
Processing S18_Sess03_FB038.pkl from s42_n56/train...
Processing S23_Sess05_FB071.pkl from s42_n56/train...
Processing S02_Sess05_FB058.pkl from s42_n56/train...
Processing S16_Sess04_FB011.pkl from s42_n56/train...
Processing S07_Sess03_FB011.pkl from s42_n56/train...
Processing S07_Sess03_FB005.pkl from s42_n56/train...
Processing S16_Sess04_FB005.pkl from s42_n56/train...
Processing S11_Sess01_FB038.pkl from s42_n56/train...
Processing S06_Sess04_FB003.